In [13]:
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate

load_dotenv()

True

In [14]:
def doc_loader(file_path):
    loader = DirectoryLoader(
        file_path,
        glob="**/*.pdf",
        loader_cls=PyMuPDFLoader,
        show_progress=True
    )
    documents = loader.load()
    return documents

In [15]:
def text_splitter(documents,chunk_size=1000,chunk_overlap=150):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
        )
    chunks = splitter.split_documents(documents)
    print(f"Split {len(documents)} Documents into {len(chunks)} Chunks")
    return chunks

In [16]:
def vectorDB(document):
    try:
        print("Creating Embeddings...")
        embeddings=HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )

        print("Embedding model loaded successfully.")
        vectorstore=Chroma.from_documents(
            documents=document,
            embedding=embeddings,
            persist_directory="../data/sololeveling_vectorDB",
            collection_name="sololeveling"
        )

        retriever = vectorstore.as_retriever()
        print("Vector store created successfully.")
        return retriever
    except Exception as e:
        print(f'Embedding-Model not found: {e}')
        return None

In [17]:
def llm_func(vector):
    print("Loading LLM model...")
    llm = ChatOpenAI(model="moonshotai/kimi-k2.6:free", 
                 api_key=os.getenv("OPENROUTER_API_KEY"),
                 base_url="https://openrouter.ai/api/v1",
                 timeout=180
                 )
    print("Loading LLM model successfully")
    
    user_input = input("Ask a question based on solo leveling:")

    retriver = vector.invoke(user_input)

    context="\n".join(
        doc.page_content for doc in retriver
        )
    
    prompt = PromptTemplate(
        template="""
        Use the give context to answer the following question
        context:{context}
        question:{question}
        """,
        input_variables=["context","question"]
    )
    final_prompt = prompt.format(context=context,question=user_input)
    return llm.invoke(final_prompt)

In [18]:
def main():
    documents = doc_loader("../data")
    chunks = text_splitter(documents)
    retriever = vectorDB(chunks)
    answer = llm_func(retriever)
    print("Answer:", answer.content)
if __name__ == "__main__":
    main()

100%|██████████| 3/3 [00:02<00:00,  1.33it/s]


Split 781 Documents into 1573 Chunks
Creating Embeddings...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1502.23it/s]


Embedding model loaded successfully.
Vector store created successfully.
Loading LLM model...
Loading LLM model successfully
Answer: Based on the context provided, here is the information about **Jinwoo**:

**Role and Relationship**
- He is Jinho's **boss**.
- They recently moved into a new office together two days prior to this scene.

**Abilities and Skills**
- Jinwoo possesses **extraordinary stealth**. When he conceals his presence, others cannot sense him even when he is standing directly beside them.
- His stealth abilities are **improving by the day**.

**Recent Fame**
- The world is currently **obsessing over him** following his involvement in the **Jeju Island raid**.

**Personality and Demeanor**
- He is **casual and calm** in his interactions, remaining relaxed even when Jinho is startled by his sudden appearance.
- He startled Jinho by silently appearing beside him and asking, "Why so serious?"

**Other Notes**
- Jinho was considering naming their guild the **"WooHo Guild"**